# Design a Distributed Task Scheduler

**Company:** MongoDB (GothamLoop question bank) · **Category:** System Design · **Tags:** Onsite Loop, Caching, Concurrency, Databases, Distributed Systems · **Difficulty/Frequency:** Uncommon (3/10)

> **Related:** [`18. Retry_Strategy`](../../2.%20Coding_Questions/18.%20Retry_Strategy/18.%20Retry_Strategy.ipynb) in the coding folder is this problem's backoff logic at single-process scale, and [`11. Task_Scheduler`](../../2.%20Coding_Questions/11.%20Task_Scheduler/11.%20Task_Scheduler.ipynb) is its dependency-ordering half.

## Concepts

**What this question is really testing:**
- **Claiming work safely** when several schedulers race for the same rows
- **Leases**: how a system notices that a worker died, without the worker telling it
- Knowing that **exactly-once does not exist**, and what you offer instead

**First-principles primer — what is each piece?**

- **The brain/brawn split.** A cron daemon does both — decides *what* runs and *runs* it — so one slow task blocks every other schedule. Separating them means the scheduler only ever does bookkeeping (fast, bounded) while workers do the arbitrary, slow, failure-prone work.
- **A lease.** A claim with an **expiry**. The worker says "this is mine until 14:03:22" and must keep re-saying it. That inversion is the whole trick: a crashed worker cannot tell you it crashed, but it *also* cannot renew, and silence is the signal.
- **`FOR UPDATE SKIP LOCKED`.** Ordinary `SELECT ... FOR UPDATE` makes concurrent schedulers **queue behind each other** on the same rows — correct, but serialised. `SKIP LOCKED` says *"anything already locked, pretend it isn't there"*, so N schedulers claim N disjoint batches simultaneously with no coordination protocol at all. The database's row locks *are* the distributed lock.
- **Partial index.** `WHERE status IN ('pending','failed')` means the index contains only claimable rows. A table of 100M completed tasks has an index of a few thousand — which is why the claim query stays fast as history accumulates.

**The guarantee, stated honestly:**

> **At-least-once.** A task will never be lost. It may run twice.

Duplicates happen in a specific, unavoidable window: a worker finishes the real work, and its lease expires *before* it writes the result. Another worker picks the task up and does it again. You cannot close that window — the work and the acknowledgement are two separate operations, and any crash between them is indistinguishable from a crash before them.

So the contract is: **the system guarantees delivery; the task owner guarantees idempotency.** Saying that out loud is the answer.

**The three failure points, and what covers each:**

| What dies | How it is noticed | Recovery |
|---|---|---|
| Scheduler, mid-claim | the claim transaction never commits | nothing was claimed; next loop picks it up |
| Scheduler, after commit, before publishing | lease expires with nobody working | re-claimed after `lease_ttl` |
| Worker, mid-execution | lease stops being renewed | re-claimed after `lease_ttl`, `attempt_count` grows |

**Simple worked example.** A task due at 09:00, `lease_ttl` 30s, renewed every 10s:

```
09:00:00  scheduler claims it   -> status=running, lease_expires=09:00:30, attempt=1
09:00:00  published to queue
09:00:01  worker picks it up, starts a 5-minute job
09:00:10  worker renews         -> lease_expires=09:00:40
09:00:20  worker renews         -> lease_expires=09:00:50
09:00:25  *** worker's machine loses power ***
09:00:50  lease expires. Nobody renewed it.
09:00:51  scheduler's claim query sees lease_expires < now() -> RE-CLAIMS
                                 -> attempt=2, a different worker starts over
```

The dead worker never reported anything. It did not have to.

## Requirements & Scale

| Functional | Non-functional |
|---|---|
| CRUD API for tasks | Scheduler is **not** a single point of failure |
| One-time **and** cron schedules | Adding workers scales throughput **linearly** |
| Status tracking + execution history | Low operational overhead |
| Configurable retries and backoff | **At-least-once** execution |

**Stated scale:** "millions of tasks per day" — we take 5M/day and derive the rest.

In [ ]:
import os, sys
_root = os.getcwd()
for _ in range(5):
    if os.path.exists(os.path.join(_root, "capacity.py")):
        break
    _root = os.path.dirname(_root)
if _root not in sys.path:
    sys.path.insert(0, _root)

from capacity import (DAY, KB, MB, GB, TB, human_bytes, human_count,
                      human_rate, table, assumption_table, sensitivity)

TASKS_PER_DAY = 5_000_000
PEAK_MULTIPLIER = 10          # the answer says "peak might be 10x"
CLAIM_BATCH = 100             # LIMIT in the claim query
CLAIM_INTERVAL_MS = 100       # how often each scheduler runs the loop
SCHEDULER_NODES = 3
TASK_CPU_MS = 100             # how long an average task takes
TASK_ROW_BYTES = 500
EXECUTION_ROW_BYTES = 200
AVG_ATTEMPTS = 1.2
RETENTION_DAYS = 30
LEASE_TTL_SEC = 30
RENEW_INTERVAL_SEC = 10

assumption_table({
    "Tasks per day":          human_count(TASKS_PER_DAY),
    "Peak multiplier":        f"{PEAK_MULTIPLIER}x average",
    "Claim batch size":       CLAIM_BATCH,
    "Claim loop interval":    f"{CLAIM_INTERVAL_MS} ms",
    "Scheduler nodes":        SCHEDULER_NODES,
    "Avg task duration":      f"{TASK_CPU_MS} ms",
    "Avg attempts per task":  AVG_ATTEMPTS,
    "Retention":              f"{RETENTION_DAYS} days",
    "Lease TTL / renew":      f"{LEASE_TTL_SEC}s TTL, renewed every {RENEW_INTERVAL_SEC}s",
})

## Capacity model — and finding the real bottleneck

Three capacities to compare. The interesting result is that the component everyone worries about (the database) has enormous headroom, and the one nobody mentions (workers) is the actual constraint.

In [ ]:
# ---- Demand ---------------------------------------------------------------
tps_avg = TASKS_PER_DAY / DAY
tps_peak = tps_avg * PEAK_MULTIPLIER

table([
    ("Average task rate", human_rate(tps_avg, " TPS")),
    (f"Peak ({PEAK_MULTIPLIER}x)", human_rate(tps_peak, " TPS")),
], title="DEMAND")

assert 55 < tps_avg < 60, "the answer's stated ~58 TPS"
assert 550 < tps_peak < 600, "the answer's stated ~580 TPS"

# ---- Supply 1: how much can the SCHEDULERS claim? -------------------------
claims_per_sec_per_node = CLAIM_BATCH * (1000 / CLAIM_INTERVAL_MS)
claim_capacity = claims_per_sec_per_node * SCHEDULER_NODES

# ---- Supply 2: how much can the WORKERS execute? --------------------------
tasks_per_worker_per_sec = 1000 / TASK_CPU_MS
workers_needed = tps_peak / tasks_per_worker_per_sec

table([
    ("Claim capacity per scheduler", human_rate(claims_per_sec_per_node, " TPS")),
    (f"Claim capacity, {SCHEDULER_NODES} nodes", human_rate(claim_capacity, " TPS")),
    ("Headroom over peak",           f"{claim_capacity / tps_peak:.1f}x"),
    ("", ""),
    ("Tasks per worker per sec",     f"{tasks_per_worker_per_sec:.0f}"),
    ("Workers needed at peak",       f"{workers_needed:.0f}"),
], title="SUPPLY")

assert claim_capacity == 3000, "the answer's stated 3,000 TPS of claim capacity"
assert 55 < workers_needed < 65, "the answer's stated ~60 workers"

print(f"\n  => The DATABASE has {claim_capacity / tps_peak:.0f}x headroom.")
print(f"     The WORKERS are the bottleneck - and they scale linearly, which is the point.")

In [ ]:
# ---- Storage --------------------------------------------------------------
task_bytes_per_day = TASKS_PER_DAY * TASK_ROW_BYTES
exec_rows_per_day = TASKS_PER_DAY * AVG_ATTEMPTS
exec_bytes_per_day = exec_rows_per_day * EXECUTION_ROW_BYTES

table([
    ("tasks rows / day",        human_bytes(task_bytes_per_day)),
    ("task_executions / day",   human_bytes(exec_bytes_per_day)),
    ("Total / day",             human_bytes(task_bytes_per_day + exec_bytes_per_day)),
    ("", ""),
    (f"Hot set ({RETENTION_DAYS}d)",
     human_bytes((task_bytes_per_day + exec_bytes_per_day) * RETENTION_DAYS)),
    ("Unbounded (1 year)",
     human_bytes((task_bytes_per_day + exec_bytes_per_day) * 365)),
], title="STORAGE")

assert abs(task_bytes_per_day - 2.5 * GB) / GB < 0.01, "the answer's stated 2.5 GB/day"
assert abs(exec_bytes_per_day - 1.2 * GB) / GB < 0.01, "the answer's stated 1.2 GB/day"

print("\n  => Archiving is not an optimisation, it is what keeps the PARTIAL INDEX small.")
print("     Without it the claim query's index grows forever, even though the number of")
print("     CLAIMABLE rows never does.")

# ---- Why the partial index matters ----------------------------------------
hot_rows = tps_peak * 60          # roughly a minute's worth of due tasks
one_year_rows = TASKS_PER_DAY * 365
table([
    ("Rows in the table after 1 year", human_count(one_year_rows)),
    ("Rows the claim query needs",     human_count(hot_rows)),
    ("Ratio",                          f"{one_year_rows / hot_rows:,.0f} : 1"),
], title="WHY A PARTIAL INDEX")
print("\n  => A full index would carry 1.8B entries to find ~35,000 relevant ones.")

## The claim query — the heart of the design

```sql
UPDATE tasks
SET status = 'running',
    lease_owner = :scheduler_id,
    lease_expires_at = now() + interval '30 seconds',
    attempt_count = attempt_count + 1
WHERE task_id IN (
    SELECT task_id FROM tasks
    WHERE next_run_at <= now()
      AND status IN ('pending', 'failed')
      AND (lease_expires_at IS NULL OR lease_expires_at < now())   -- <- recovers dead workers
    ORDER BY next_run_at
    LIMIT 100
    FOR UPDATE SKIP LOCKED                                          -- <- lets schedulers run concurrently
)
RETURNING *;                                                        -- <- claim and read in ONE transaction
```

Four clauses, four distinct jobs:

| Clause | Without it |
|---|---|
| `FOR UPDATE SKIP LOCKED` | schedulers block on each other; only one makes progress |
| `lease_expires_at < now()` | a crashed worker's task is stuck in `running` forever |
| `RETURNING *` | you need a second read, and a crash between them loses the claim |
| the partial index | full table scan, every 100 ms, forever |

**Why no leader election?** Because the claim is already atomic. Consensus protocols exist to decide *who may act*; here the database decides that per-row, for free. Adding ZooKeeper would be strictly worse: more moving parts, and a leader is a bottleneck where `SKIP LOCKED` is a parallelism *enabler*.

### ⚠️ As written, the recovery clause can never fire

Read the two predicates together:

```sql
AND status IN ('pending', 'failed')
AND (lease_expires_at IS NULL OR lease_expires_at < now())
```

The `UPDATE` sets `status = 'running'`. So a task held by a worker that then dies is in `running` — and `running` is not in `('pending','failed')`. **The row is filtered out before the lease check is ever reached.** The expiry clause is dead code, and the crashed task is stranded forever.

The status set has to admit expired-but-running rows:

```sql
AND (
      status IN ('pending', 'failed')
   OR (status = 'running' AND lease_expires_at < now())   -- the actual recovery path
)
```

The cell below asserts both: the query as printed strands the task, and the corrected one recovers it. This is worth spotting out loud in an interview — the lease *mechanism* is right, and the *query* silently doesn't use it.

In [ ]:
# A model of the claim, to make the concurrency argument concrete rather than asserted.
class TaskRow:
    def __init__(self, tid, due_at, status="pending", attempt=0, lease_until=None):
        self.tid, self.due_at, self.status = tid, due_at, status
        self.attempt, self.lease_until = attempt, lease_until
        self.locked_by = None

    def claimable(self, now, recover_running=True):
        if self.due_at > now or self.locked_by is not None:   # locked_by <- SKIP LOCKED
            return False
        if self.status in ("pending", "failed"):
            return self.lease_until is None or self.lease_until < now
        if recover_running and self.status == "running":
            return self.lease_until is not None and self.lease_until < now
        return False


def claim_batch(rows, scheduler_id, now, batch=CLAIM_BATCH, lease_ttl=LEASE_TTL_SEC,
                recover_running=True):
    """One scheduler's claim. Returns the rows it won."""
    won = []
    for r in sorted((r for r in rows if r.claimable(now, recover_running)),
                    key=lambda r: r.due_at):
        if len(won) >= batch:
            break
        r.locked_by = scheduler_id                   # the row lock
        r.status = "running"
        r.lease_until = now + lease_ttl
        r.attempt += 1
        won.append(r)
    for r in won:
        r.locked_by = None                           # transaction commits, lock released
    return won


# Three schedulers race for 250 due tasks. No task may be claimed twice.
rows = [TaskRow(i, due_at=0) for i in range(250)]
batches = [claim_batch(rows, f"sched-{i}", now=1) for i in range(3)]

claimed = [r.tid for b in batches for r in b]
assert len(claimed) == len(set(claimed)), "SKIP LOCKED must prevent double-claiming"
assert len(claimed) == 250, f"all due tasks claimed exactly once, got {len(claimed)}"
table([(f"scheduler-{i}", f"claimed {len(b)}") for i, b in enumerate(batches)]
      + [("total, no duplicates", str(len(claimed)))],
      title="THREE SCHEDULERS, ONE BATCH EACH")

# Now: a worker dies. The lease expires and the task should come back.
t = TaskRow(999, due_at=0)
claim_batch([t], "sched-0", now=100)
assert t.status == "running" and t.attempt == 1 and t.lease_until == 130

assert claim_batch([t], "sched-1", now=110) == [], "a LIVE lease must block re-claiming"

# The query AS PRINTED: status IN ('pending','failed') excludes the running row.
assert claim_batch([t], "sched-1", now=131, recover_running=False) == [], \
    "the source's query strands the task - status is 'running', so it never matches"
assert t.attempt == 1, "nothing happened; the task is lost"
print("\n  As printed, at t=131 (lease long expired): NOT re-claimed. Task stranded.")

# The corrected query: expired-but-running rows are claimable.
recovered = claim_batch([t], "sched-1", now=131, recover_running=True)
assert len(recovered) == 1 and t.attempt == 2, "an EXPIRED lease must allow re-claiming"
print("  Corrected, same instant:                  re-claimed, attempt=2.")

### The subtlety: `attempt_count` counts *claims*, not *executions*

The claim query increments `attempt_count`. That is what stops a task retrying forever — but it means **a task that is claimed and never executed still burns a retry**.

Scenario: the scheduler commits the claim, then crashes before publishing to the queue. The task waits out its lease, is re-claimed, and `attempt_count` is now 2 — for a task that has never run once. Three such crashes exhaust `max_retries = 3` on a task that was never attempted.

The fix is to separate the two counters: keep the retry budget tied to **executions actually started** (recorded in `task_executions`), and track claims separately for observability.

In [ ]:
def simulate(max_retries=3, scheduler_crashes=0, execution_failures=0,
             count_claims_as_attempts=True):
    """Does the task survive? Returns (outcome, executions_started)."""
    attempts, executions = 0, 0
    for _ in range(20):                      # bounded loop
        if attempts >= max_retries:
            return ("GAVE UP (never ran!)" if executions == 0
                    else "GAVE UP (after real failures)"), executions
        if count_claims_as_attempts:
            attempts += 1                    # naive: every CLAIM costs a retry
        if scheduler_crashes > 0:
            scheduler_crashes -= 1
            continue                         # claimed, never published; lease expires
        if not count_claims_as_attempts:
            attempts += 1                    # the fix: only a real EXECUTION costs one
        executions += 1
        if execution_failures > 0:
            execution_failures -= 1
            continue
        return "SUCCEEDED", executions
    return "LOOPED", executions


# The bug: three scheduler crashes exhaust a task that never executed.
outcome, execs = simulate(max_retries=3, scheduler_crashes=3)
assert outcome == "GAVE UP (never ran!)" and execs == 0, (outcome, execs)
print(f"  Naive (claims count):      {outcome:<32} executions={execs}")

# The fix: the same three crashes, but only executions count against the budget.
outcome, execs = simulate(max_retries=3, scheduler_crashes=3,
                          count_claims_as_attempts=False)
assert outcome == "SUCCEEDED" and execs == 1, (outcome, execs)
print(f"  Fixed (executions count):  {outcome:<32} executions={execs}")

# Genuine failures must still exhaust the budget - the fix must not break retries.
outcome, execs = simulate(max_retries=3, execution_failures=5,
                          count_claims_as_attempts=False)
assert outcome == "GAVE UP (after real failures)", outcome
print(f"  Real failures still stop:  {outcome:<32} executions={execs}")

## Recurring schedules, and why DST is not a footnote

The naive approach — store the interval, add it each run — breaks on the two days a year that do not have 24 hours, and on every month that is not 30 days.

The robust approach stores the **cron expression** and the **IANA timezone** (`America/New_York`, never `EST`), and recomputes the next UTC instant from scratch after each run.

Two days a year force a policy decision you should state rather than discover:

| Event | What happens to a 02:30 daily task | Policy |
|---|---|---|
| **Spring forward** — 02:00 jumps to 03:00 | 02:30 does not exist | run at 03:00, **or** skip that day |
| **Fall back** — 01:00–02:00 happens twice | 01:30 exists twice | run **once**, on the first occurrence |

Storing an offset (`UTC-5`) instead of a zone name is the classic bug: the offset is only correct for half the year.

In [ ]:
from datetime import datetime, date, timedelta, tzinfo, timezone as _tz

# In production you would write `ZoneInfo("America/New_York")` and stop. Here we
# implement the rule by hand, both so the notebook runs without the tzdata package
# and to make concrete what a tz database entry actually *is*: a pair of switchover
# rules plus two offsets.
try:
    from zoneinfo import ZoneInfo
    NY = ZoneInfo("America/New_York")
    NY.utcoffset(datetime(2024, 7, 1))          # raises if tzdata is missing
    SOURCE = 'zoneinfo.ZoneInfo("America/New_York")'
except Exception:                               # no tzdata installed
    def _nth_sunday(year, month, n):
        first = date(year, month, 1)
        return 1 + (6 - first.weekday()) % 7 + 7 * (n - 1)

    class USEastern(tzinfo):
        """US rule since 2007: DST from the 2nd Sunday in March to the 1st in November."""
        STD, DST = timedelta(hours=-5), timedelta(hours=-4)

        def _is_dst(self, dt):
            if dt is None:
                return False
            naive = dt.replace(tzinfo=None)
            start = datetime(dt.year, 3, _nth_sunday(dt.year, 3, 2), 2, 0)
            end = datetime(dt.year, 11, _nth_sunday(dt.year, 11, 1), 2, 0)
            if naive < start or naive >= end:
                return False
            # The hour before `end` happens twice; fold=1 is the second (standard) pass.
            if end - timedelta(hours=1) <= naive and dt.fold:
                return False
            return True

        def utcoffset(self, dt): return self.DST if self._is_dst(dt) else self.STD
        def dst(self, dt): return timedelta(hours=1) if self._is_dst(dt) else timedelta(0)
        def tzname(self, dt): return "EDT" if self._is_dst(dt) else "EST"

    NY = USEastern()
    SOURCE = "hand-rolled US rule (tzdata not installed)"

print(f"  Using: {SOURCE}\n")

# Spring forward 2024: 02:00 EST -> 03:00 EDT on 10 March. 02:30 does not exist.
before = datetime(2024, 3, 10, 1, 30, tzinfo=NY)
after = before + timedelta(hours=1)
table([
    ("01:30 EST, +1 hour", after.strftime("%H:%M %Z")),
    ("UTC offset before",  before.strftime("%z")),
    ("UTC offset after",   after.strftime("%z")),
], title="SPRING FORWARD (10 Mar 2024, America/New_York)")
assert before.utcoffset() != after.utcoffset(), "the offset CHANGES mid-day"

# The bug: a fixed offset is right for only half the year.
winter = datetime(2024, 1, 15, 12, 0, tzinfo=NY)
summer = datetime(2024, 7, 15, 12, 0, tzinfo=NY)
table([
    ("Noon in January", f"UTC{winter.strftime('%z')}  ({winter.tzname()})"),
    ("Noon in July",    f"UTC{summer.strftime('%z')}  ({summer.tzname()})"),
], title="WHY YOU STORE A ZONE NAME, NOT AN OFFSET")
assert winter.utcoffset() != summer.utcoffset()
print("\n  => Storing 'UTC-5' would run every summer task an hour late.")
print("     Store 'America/New_York' and recompute; never store the offset.")

# Fall back: 01:30 occurs TWICE. Must fire once.
fall_back_first = datetime(2024, 11, 3, 1, 30, tzinfo=NY, fold=0)
fall_back_second = datetime(2024, 11, 3, 1, 30, tzinfo=NY, fold=1)
assert fall_back_first.utcoffset() != fall_back_second.utcoffset()
delta_hours = (fall_back_second.astimezone(_tz.utc)
               - fall_back_first.astimezone(_tz.utc)).total_seconds() / 3600
assert delta_hours == 1.0
print(f"\n  Fall back: 01:30 exists twice, {delta_hours:.0f} hour apart in UTC.")
print("  Dedupe on the computed UTC instant, or the task fires twice.")

## Discussion — the follow-ups

- **Task dependencies (a DAG).** Add `depends_on` and refuse to claim a task until every dependency is `succeeded`. The claim query gains a `NOT EXISTS (SELECT 1 FROM deps WHERE ... AND status <> 'succeeded')`. What you have then built is [Kahn's algorithm](../../2.%20Coding_Questions/11.%20Task_Scheduler/11.%20Task_Scheduler.ipynb) with the in-degree kept in a database rather than a dict — and the same cycle problem applies, so validate the DAG at submission time rather than discovering a deadlock at run time.
- **A one-time task whose moment has passed.** Genuinely ambiguous, so make it a field: `missed_run_policy` of `run_immediately` (a nightly report — you still want it), `skip` (a "meeting starts now" alert — firing it late is worse than not firing), or `run_once_late` with a staleness bound. The wrong move is to pick silently; the *user* knows which their task is.
- **Per-user flooding.** Two layers: a **rate limit** at the API (submissions per minute) and a **quota** in the database (active tasks per user). Note that only the quota protects the *scheduler* — a user can submit slowly and still accumulate a million tasks all due at 09:00. Better still, add a per-user cap on the claim query itself, so one tenant cannot monopolise a batch.
- **Priorities.** Order the claim by `(priority, next_run_at)` and it mostly works — but strict priority **starves** low-priority tasks under sustained high-priority load. Separate queues with weighted consumption (e.g. workers pull 7:2:1 from high/normal/low) bounds the starvation. This is the same fairness trade as the [read/write lock](../../2.%20Coding_Questions/16.%20Read_Write_Lock/16.%20Read_Write_Lock.ipynb).
- **A 3-hour task whose worker gets partitioned.** The lease expires, the task is reassigned, and **it now runs twice concurrently** — the genuinely nasty case, because both workers believe they hold it. Mitigations, in increasing order of strength: lengthen the lease for long tasks (narrows the window, does not close it); have the worker **re-verify it still owns the lease before writing the result** (`UPDATE ... WHERE lease_owner = :me`, so the loser's write is rejected); or make the task idempotent, which is the only actual fix. This is a **fencing token** problem — the same one behind distributed locks generally.

## Patterns learned

- **Separate deciding from doing.** The scheduler does bounded bookkeeping; workers do unbounded, failure-prone work. One slow task then cannot block every other schedule.
- **A lease turns silence into a signal.** A crashed process cannot report its own death, but it also cannot renew. Expiry *is* the detection mechanism — no heartbeat protocol, no failure detector, no consensus.
- **Let the database be the distributed lock.** `FOR UPDATE SKIP LOCKED` gives N schedulers disjoint work with no coordination protocol. Reaching for ZooKeeper here adds a bottleneck where the database offered parallelism.
- **Claim and read in one transaction.** `UPDATE ... RETURNING` leaves no window where a crash loses a claim you have already made.
- **Index only what you query.** A partial index on claimable rows stays tiny while the table grows without bound — which is why archiving is a *correctness-adjacent* concern, not just cost control.
- **At-least-once is what distributed systems offer.** Exactly-once requires the work and its acknowledgement to be atomic, and they are not. The system guarantees delivery; the task owner guarantees idempotency.
- **Count the thing you actually mean.** Incrementing the retry budget on *claim* rather than *execution* quietly exhausts tasks that never ran.
- **Check that your recovery path is reachable.** The lease design was right and the query filtered `status = 'running'` out before the expiry check — a mechanism that looks correct and never runs. Trace one concrete failure through the actual predicate.
- **Store the zone, never the offset.** `America/New_York`, recomputed each time. An offset is correct for half the year.
- **Find the real bottleneck before optimising.** Here the database has 5× headroom and the workers do not — so the scaling story is "add workers", and it is linear.